In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, row_number

# 1. Read the raw data from your Bronze table
df_bronze = spark.read.table('deve.bronze.orders')

# 2. Cleanse and Transform
# - Cast the 'amount' string to a double (decimal number)
# - Cast the 'order_date' string to an actual Date type
# - Filter out any negative amounts (business rule)
df_clean = df_bronze.withColumn('amount', col('amount').cast('double')) \
                    .withColumn('order_date', to_date(col('order_date'))) \
                    .filter(col('amount') > 0)

# -------------------------------------------------------------------
# THE FIX: Deduplicate source data before merging
# Group by order_id, sort by the time it was ingested (newest first)
window_spec = Window.partitionBy("order_id").orderBy(desc("ingest_time"))

# Number the rows (1 is the newest) and keep ONLY row #1
df_latest = df_clean.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1) \
                    .drop("rn")
# -------------------------------------------------------------------

# 3. Define your target Silver table
silver_table_name = "deve.silver.orders"

# 4. Upsert (MERGE) logic
if spark.catalog.tableExists(silver_table_name):
    # If the table already exists, perform a MERGE
    # This updates existing orders and inserts new ones based on order_id
    silver_table = DeltaTable.forName(spark, silver_table_name)
    silver_table.alias('target').merge(
        df_latest.alias('source'),
        'target.order_id = source.order_id'
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f'Successfully created {silver_table_name}')
else:
    # If this is the very first run, create the table
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    print(f"Successfully created {silver_table_name}")

# 5. Show the resulting clean data
display(spark.read.table(silver_table_name))